Repurposed a lot of code from the first version of my imposition algorithm, that took the output from bookbinder.app, and made this.

As stated in the first version of this new version of things, the main reason for this is to circumvent possible errors that the first method may have, specially due to the webapps not processing the pdfs properly. Which means that this will make sure that everything works correctly, fast, with minimum file bloat, and won't give any errors... probably.

The main logic was solved by me. I figured that all I had to do was to get to the same output I needed to get to in order to do the imposition on the "WayTowardsPrinting" algorithm. I didn't want to figure out the signature order and asked DeepSeek to do it (it had already done from the first time, I just took the code and it worked here :P). The only other vibe-coded thing was the shuffle_join function, which I wanted just so that I didn't need to pull another library to use the old method I used in the first algorithm.

Either way, it's finally here! So finally there's two versions of this, both usable, fast, and giving minimum file bloat.

If you found this, I hope this is all useful to you and works better than whatever else you were using before. Next thing I might try to implement is a manga/japanese/reverse signature order, in case someone wants to print that. Either way, ggs.

In [ ]:
book_name = "book_name"

In [2]:
def imposition(n: int, pages_per_signature: int = 20):
    """
    Return (fronts, backs) for printing a book of n pages.
    pages_per_signature should be a multiple of 4 (default 20 = 5 sheets).
    The last signature is padded with None to a multiple of 4.
    """
    fronts = []
    backs = []
    start = 1

    while start <= n:
        remaining = n - start + 1
        size = min(pages_per_signature, remaining)
        # Pad to next multiple of 4
        padded_size = ((size + 3) // 4) * 4
        pages = list(range(start, start + size)) + [None] * (padded_size - size)

        # Build sheets for this signature
        for i in range(padded_size // 4):
            # local 1-based indices mapped to 0-based list
            fl = pages[padded_size - 2*i - 1]   # front left
            fr = pages[1 + 2*i - 1]             # front right
            bl = pages[2 + 2*i - 1]             # back left
            br = pages[padded_size - 1 - 2*i - 1]  # back right

            fronts.extend([fl, fr])
            backs.extend([bl, br])

        start += size

    return fronts, backs

In [3]:
import pymupdf

src = pymupdf.open(f"{book_name}.pdf")
n = len(src)

twoUP_front = pymupdf.open() 
twoUP_back = pymupdf.open()

## setting up an A5 page to impose the pages ##
width, height = pymupdf.paper_size("a5")
r = pymupdf.Rect(0, 0, width, height)

# gotta define 2 rectangles, one ends up in the middle and the other is the rest #
r1 = pymupdf.Rect(0,0,width, height/2)  # top rect
r2 = pymupdf.Rect(0, height/2, width, height) # bottom rect
r_tab = [r1, r2] # thankfully i did it before :>

front, back = imposition(n)

def make_doc(doc, index):
    k=0
    for pno in index:
        if k % 2 == 0: # create new output page
                page = doc.new_page(-1, width = width, height = height)
        if pno==None:
                k+=1
                continue
        else:
                x=pno-1
        page.show_pdf_page(r_tab[k % 2], src, x, rotate=-90) # insert input page into correct rectangle rotated 90
        k+=1


front, back = imposition(n)
make_doc(twoUP_front, front)
make_doc(twoUP_back, back)

twoUP_front.save("pdfF.pdf", garbage=4, deflate=True, compression_effort=100)
twoUP_back.save("pdfB.pdf",  garbage=4, deflate=True, compression_effort=100)

In [4]:
src = pymupdf.open("pdfF.pdf") # open front only pdf
doc = pymupdf.open()

width, height = pymupdf.paper_size("a4")

r1 = pymupdf.Rect(0,0,width, height/2)  # top rect
r2 = pymupdf.Rect(0, height/2, width, height) # bottom rect
r_tab = [r1, r2]

for spage in src:
    if spage.number % 2 == 0: # create new output page
            page = doc.new_page(-1, width = width, height = height)
    page.show_pdf_page(r_tab[spage.number % 2], src, spage.number, rotate=90) # insert input page into correct rectangle rotated 90

doc.save("2upF.pdf", garbage=3, deflate=True)

src = pymupdf.open("pdfB.pdf") # open back only pdf
doc = pymupdf.open()

for spage in src:
    if spage.number % 2 == 0:
            page = doc.new_page(-1, width = width, height = height)
    page.show_pdf_page(r_tab[spage.number % 2], src, spage.number, rotate=90)

doc.save("2upB.pdf", garbage=3, deflate=True)

def shuffle_join(pdf_a, pdf_b, output):
    a = pymupdf.open(pdf_a)
    b = pymupdf.open(pdf_b)
    out = pymupdf.open()

    # interleave: A1, B1, A2, B2, ...
    for i in range(max(len(a), len(b))):
        if i < len(a):
            out.insert_pdf(a, from_page=i, to_page=i)
        if i < len(b):
            out.insert_pdf(b, from_page=i, to_page=i)

    out.save(output, garbage=3, deflate=True)

shuffle_join("2upF.pdf", "2upB.pdf", f"[final] {book_name}.pdf")